# Gomoku 9x9 — Train Heuristic Predictor ANN
Dùng GPU của Colab để train ANN xấp xỉ heuristic truyền thống.

**Output:** `heuristic_predictor.onnx` + `scaler.pkl` → tải về máy, bỏ vào `models/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch onnx onnxruntime scikit-learn joblib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.6 MB/s eta 0:00:00


In [3]:
from google.colab import files
print('Upload X_data.npy and y_data.npy...')
uploaded = files.upload()

Upload X_data.npy and y_data.npy...


Saving X_data.npy to X_data.npy
Saving y_data.npy to y_data.npy


In [4]:
import torch
import torch.nn as nn

INPUT_SIZE = 81
HIDDEN1 = 256
HIDDEN2 = 128
HIDDEN3 = 64

class HeuristicPredictor(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, hidden1=HIDDEN1, hidden2=HIDDEN2, hidden3=HIDDEN3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden1),
            nn.Dropout(0.3),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden2),
            nn.Dropout(0.3),
            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.BatchNorm1d(hidden3),
            nn.Linear(hidden3, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

DATA_DIR = '/content/'
MODELS_DIR = '/content/drive/MyDrive/gomoku_models'
os.makedirs(MODELS_DIR, exist_ok=True)

print('[Train] Loading data...')
X = np.load(os.path.join(DATA_DIR, 'X_data.npy'))
y = np.load(os.path.join(DATA_DIR, 'y_data.npy'))
print(f'  X: {X.shape}, y: {y.shape}')

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)
print(f'  Train: {len(X_train)}, Val: {len(X_val)}')

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).view(-1, 1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val).view(-1, 1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Train] Using device: {device}')

model = HeuristicPredictor().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

X_train_t = X_train_t.to(device)
y_train_t = y_train_t.to(device)
X_val_t = X_val_t.to(device)
y_val_t = y_val_t.to(device)

EPOCHS = 200
BATCH_SIZE = 128
best_val_loss = float('inf')

print('[Train] Starting training...')
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(X_train_t))
    epoch_loss = 0.0
    num_batches = 0

    for i in range(0, len(X_train_t), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        X_batch = X_train_t[idx]
        y_batch = y_train_t[idx]

        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, y_val_t).item()

    train_loss = epoch_loss / num_batches
    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'best_model.pth'))

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f'  Epoch {epoch:3d}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}')

print(f'[Train] Best val_loss: {best_val_loss:.6f}')

[Train] Loading data...
  X: (50000, 81), y: (50000,)
  Train: 42500, Val: 7500
[Train] Using device: cuda
[Train] Starting training...
  Epoch   0: train_loss=0.019473, val_loss=0.006482
  Epoch  20: train_loss=0.001242, val_loss=0.000324
  Epoch  40: train_loss=0.000835, val_loss=0.000362
  Epoch  60: train_loss=0.002750, val_loss=0.000753
  Epoch  80: train_loss=0.000886, val_loss=0.000038
  Epoch 100: train_loss=0.000599, val_loss=0.000831
  Epoch 120: train_loss=0.000279, val_loss=0.000019
  Epoch 140: train_loss=0.000654, val_loss=0.000327
  Epoch 160: train_loss=0.000396, val_loss=0.000249
  Epoch 180: train_loss=0.000744, val_loss=0.000193
  Epoch 199: train_loss=0.000347, val_loss=0.000277
[Train] Best val_loss: 0.000010


In [7]:
import os
import onnx

# Install onnxscript if it's missing (a dependency for torch.onnx.export)
try:
    import onnxscript
except ImportError:
    print("Installing 'onnxscript'...")
    !pip install onnxscript
    print("'onnxscript' installed.")

model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'best_model.pth')))
model.eval()

dummy = torch.randn(1, 81).to(device)
onnx_path = os.path.join(MODELS_DIR, 'heuristic_predictor.onnx')
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['board_input'],
    output_names=['heuristic_score'],
    dynamic_axes={'board_input': {0: 'batch_size'}},
)
    model_onnx = onnx.load(onnx_path)
    onnx.save_model(model_onnx, onnx_path, save_as_external_data=False)
print(f'[Train] ONNX model saved to {onnx_path}')

scaler_path = os.path.join(MODELS_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f'[Train] Scaler saved to {scaler_path}')
print('[Train] Done!')

Installing 'onnxscript'...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.8 MB/s eta 0:00:00
'onnxscript' installed.


/tmp/ipykernel_2941/3519561340.py:16: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `HeuristicPredictor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `HeuristicPredictor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[Train] ONNX model saved to /content/drive/MyDrive/gomoku_models/heuristic_predictor.onnx
[Train] Scaler saved to /content/drive/MyDrive/gomoku_models/scaler.pkl
[Train] Done!


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [8]:
from google.colab import files
files.download(os.path.join(MODELS_DIR, 'heuristic_predictor.onnx'))
files.download(os.path.join(MODELS_DIR, 'scaler.pkl'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>